# PARIKALP 2026 — Exoplanet Dataset: EDA, Physics Validation & Feature Engineering

**Author:** Sant Kumar (Vyom) | B.E. Information Technology, UIET Panjab University

**Context:** Data Analytics Challenge (PARIKALP 2026, hosted by Astro Alliance, MANIT Bhopal) — team role: EDA, data cleaning, physics validation, and feature engineering.

**Dataset:** Exoplanet dataset — 39,913 rows × 96 columns

**Targets to predict:** `pl_rade` (planet radius) and `pl_bmasse` (planet mass)

**Final goal:** After cleaning, a regression model predicts both targets and escape velocity is derived using `v_e = sqrt(2GM/R)`

---

## How to run this notebook
1. Clone this repo and open the notebook in Jupyter (or upload it to Colab).
2. Place the competition CSV in a `data/` folder next to this notebook (or update `DATA_PATH` below).
3. Run all cells top to bottom — no other setup is required beyond `pandas`, `numpy`, and `matplotlib`.


## Step 1 — Load the Dataset


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path to the dataset — place the competition CSV inside a local `data/` folder,
# or point this at wherever you've stored it.
DATA_PATH = "data/exoplanet_dataset.csv"


In [ ]:
# Confirm the dataset file is where we expect before loading it
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH}'. Download the PARIKALP 2026 exoplanet "
        "dataset and place it in a 'data/' folder next to this notebook, or update DATA_PATH above."
    )
print('Dataset file found — loading...')


In [ ]:
# Load the main dataset
df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded successfully — shape: {df.shape}')


## Step 2 — Initial Data Inspection
Before doing anything, we first understand what we have — shape, data types, missing values, and basic statistics.

This corresponds to **Section 1.1 (Data Inspection)** of the marking scheme.

In [ ]:
# Dataset dimensions — how many rows and columns
print('Shape:', df.shape)
print('Total cells:', df.shape[0] * df.shape[1])

In [ ]:
# First 5 rows — quick visual of what the data looks like
df.head()

In [ ]:
# Data types and non-null counts per column
# Non-null count tells us how many rows have actual data (missing = 39913 - non-null count)
df.info()

In [ ]:
# Data type of every column
df.dtypes

In [ ]:
# Count of missing values per column — sorted worst to best
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

In [ ]:
pd.set_option('display.max_columns', None)
df.describe()

## Step 3 — Physics Validation & Fixing Violations
Before any modeling, we check if the data violates known physical laws.
If a value is physically impossible, it is either a data entry error or a unit mismatch.

This corresponds to **Section 1.4 (Visualization)** of the marking scheme — identifying physics law violations through visualization.

### 3.1 — `sy_dist` (System Distance in Parsecs)
**Physical rule:** Distance from Earth cannot be negative. It must be a positive value.

**Expected range:** According to the Astronomical Constants PDF, the maximum distance in this dataset is ~8,500 parsecs (Galactic Bulge regions).

**Suspected issue:** The problem statement hints that one distance parameter was logged in two different units (parsecs and light-years). 1 parsec = 3.26 light-years.

In [ ]:
# Step 1: Check the distribution of sy_dist before any fix
print('sy_dist statistics BEFORE fix:')
print(df['sy_dist'].describe())
print(f"\nNegative values: {(df['sy_dist'] < 0).sum()}")

In [ ]:
# Step 2: Visualize sy_dist BEFORE fix — histogram shows negative values clearly
plt.figure(figsize=(10,4))
df['sy_dist'].plot(kind='hist', bins=100, color='red', alpha=0.7)
plt.xlim(-35000, 35000)
plt.title('sy_dist BEFORE Fix — Note the negative values on the left')
plt.xlabel('Distance (parsecs)')
plt.ylabel('Frequency')
plt.axvline(x=0, color='black', linestyle='--', label='Zero line')
plt.legend()
plt.show()

In [ ]:
# Step 3: Fix negative values — take absolute value
# Reasoning: Distance cannot be negative. The magnitude is realistic,
# suggesting a sign entry error rather than a completely wrong value.
df['sy_dist'] = df['sy_dist'].abs()

# Step 4: Check how many values exceed the expected parsec maximum (~8504)
# These are likely recorded in light-years instead of parsecs
light_year_candidates = df[df['sy_dist'] > 8504]['sy_dist'].count()
print(f'Values above 8504 parsecs (likely in light-years): {light_year_candidates}')

In [ ]:
# Step 5: Convert values above 8504 from light-years to parsecs
# Source: Astronomical Constants PDF states 1 parsec = 3.26 light-years
# Only values above the expected max are converted — not the entire column
# This is a minimal-change strategy to avoid incorrectly converting valid extreme values
df.loc[df['sy_dist'] > 8504, 'sy_dist'] /= 3.26

print('Conversion applied. New sy_dist statistics:')
print(df['sy_dist'].describe())

In [ ]:
# Step 6: Visualize sy_dist AFTER fix — all values should now be positive and within range
plt.figure(figsize=(10,4))
df['sy_dist'].plot(kind='hist', bins=100, color='green', alpha=0.7)
plt.xlim(0, 9000)
plt.title('sy_dist AFTER Fix — All values now positive and within parsec range')
plt.xlabel('Distance (parsecs)')
plt.ylabel('Frequency')
plt.show()

### 3.2 — `pl_orbeccen` (Orbital Eccentricity)
**Physical rule:** Eccentricity must be between 0 and 1 (inclusive).
- 0 = perfectly circular orbit
- Values closer to 1 = highly elongated orbit
- Values outside 0–1 are physically impossible

**Note:** This column also has ~52% missing values — those will be handled in the imputation phase.

In [ ]:
# Check distribution
print(df['pl_orbeccen'].describe())
print(f"\nMissing values: {df['pl_orbeccen'].isna().sum()}")
print(f"Valid values (0 to 1): {df[(df['pl_orbeccen'] >= 0) & (df['pl_orbeccen'] <= 1)]['pl_orbeccen'].count()}")
print(f"Invalid values above 1: {(df['pl_orbeccen'] > 1).sum()}")
print(f"Invalid values below 0: {(df['pl_orbeccen'] < 0).sum()}")

In [ ]:
# Fix: 1 value found below 0 — no physical basis to assume true value
# No formula exists to reconstruct eccentricity from other columns
# Decision: convert to NaN and handle with missing values later
df.loc[df['pl_orbeccen'] < 0, 'pl_orbeccen'] = None
print(f'Invalid values remaining: {(df["pl_orbeccen"] < 0).sum()}')

### 3.3 — `pl_orbincl` (Orbital Inclination)
**Physical rule:** Inclination is an angle measured in degrees — valid range is 0° to 180°.
Values outside this range are physically impossible.

**Note:** This column has ~56% missing values — those will be handled in the imputation phase.

In [ ]:
# Check distribution
print(df['pl_orbincl'].describe())
print(f"\nMissing values: {df['pl_orbincl'].isna().sum()}")
print(f"Valid values (0 to 180): {df[(df['pl_orbincl'] >= 0) & (df['pl_orbincl'] <= 180)]['pl_orbincl'].count()}")
print(f"Invalid values: {((df['pl_orbincl'] < 0) | (df['pl_orbincl'] > 180)).sum()}")

In [ ]:
# Fix: 1 value found at -39 degrees — physically impossible
# No scientific basis to assume -39 means +39 (unlike sy_dist where pattern was clear)
# Decision: convert to NaN and handle with missing values later
df.loc[df['pl_orbincl'] < 0, 'pl_orbincl'] = None
print(f'Invalid values remaining: {(df["pl_orbincl"] < 0).sum()}')

### 3.4 — `st_mass` (Stellar Mass)
**Physical rule:** Mass cannot be negative — a star with negative mass has no physical meaning.

**Expected:** All values should be positive (in units of Solar masses).

In [ ]:
# Check for negative mass values
negative_mass = df[df['st_mass'] < 0]['st_mass']
print(f'Negative st_mass values found: {len(negative_mass)}')
print(negative_mass)

In [ ]:
# Fix: 3 values found with small negative mass (-0.002, -0.0003, -0.023)
# The magnitude of all 3 is realistic for a star mass
# Conclusion: sign entry error, not a fundamentally wrong measurement
# Fix: take absolute value
df['st_mass'] = df['st_mass'].abs()
print(f'Negative values remaining: {(df["st_mass"] < 0).sum()}')

### 3.5 — Other Physical Columns
Checked: `pl_orbsmax`, `pl_orbper`, `st_rad`, `st_teff`

All must be positive (distance, time period, radius, temperature in Kelvin).

**Result:** No invalid values found in any of these columns.

In [ ]:
# Verify minimum values — all should be positive
cols_to_check = ['pl_orbsmax', 'pl_orbper', 'st_rad', 'st_teff']
for col in cols_to_check:
    min_val = df[col].min()
    status = 'OK' if min_val >= 0 else 'PROBLEM'
    print(f'{col}: min = {min_val:.4f} [{status}]')

## Step 4 — Drop Useless Columns
Three categories of columns are removed before modeling:

1. **Noisy/Decoy columns** — artificially corrupted versions of real columns (name contains 'noisy' or 'noise')
2. **Metadata columns** — labels, IDs, dates — describe the record, not the physics
3. **Data leakage columns** — `pl_ratror` is mathematically derived from `pl_rade` (a target variable). Using it as a feature would let the model 'see the answer' during training, which is penalized by competition rules.

In [ ]:
# Drop all three categories in one operation
df.drop(columns=[
    # Noisy/decoy columns — corrupted versions of real columns
    'cosmic_noise_1', 'cosmic_noise_2',
    'pl_orbper_noisy', 'pl_orbsmax_noisy',

    # Metadata — no predictive value, just labels/IDs/dates
    'rowid', 'pl_name', 'hostname', 'pl_letter',
    'rowupdate', 'pl_pubdate', 'releasedate', 'disc_pubdate',
    'discoverymethod', 'disc_year', 'disc_facility', 'soltype',

    # Data leakage — pl_ratror = pl_rade / st_rad (derived from target)
    'pl_ratror', 'pl_ratrorerr1', 'pl_ratrorerr2'
], inplace=True)

print(f'Columns after dropping: {df.shape[1]}')
print(f'Rows unchanged: {df.shape[0]}')

## Step 5 — Export Cleaned Checkpoint
This is the cleaned dataset after physics validation and column removal.

**What's done:**
- `sy_dist`: negatives fixed, light-year values converted to parsecs
- `pl_orbeccen`: 1 invalid value → NaN
- `pl_orbincl`: 1 invalid value → NaN
- `st_mass`: 3 negative values → absolute value
- 19 useless columns dropped (noisy, metadata, leakage)

**What's NOT done yet (coming in next phase):**
- Missing value imputation (~52% missing in some columns)
- Feature engineering
- Outlier analysis


In [ ]:
# Final shape confirmation
print('Final dataset shape:', df.shape)
print('\nMissing values summary (top 10):')
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Export clean checkpoint CSV
os.makedirs('output', exist_ok=True)
output_path = 'output/clean_checkpoint.csv'
df.to_csv(output_path, index=False)
print(f'Checkpoint exported to: {output_path}')


## 📊 Step 6 — Correlation Analysis

**What we are doing:** Checking which columns are most related to our two targets `pl_rade` and `pl_bmasse`.

**Why:** Before feature engineering, we need to know which columns actually carry useful signal. Columns with near-zero correlation are weak predictors.

**Teammate note:** Do NOT skip this section — correlation results directly influenced which features were engineered and which columns were prioritized.

In [ ]:
# Calculate the correlation of 'pl_rade' with all other numerical columns
correlations = df.corr(numeric_only=True)['pl_rade'].sort_values(ascending=False)

# Display the top 10 positive and negative correlations (excluding self-correlation)
print('Top 10 positive correlations with pl_rade:')
print(correlations[1:11])

print('\nTop 10 negative correlations with pl_rade:')
print(correlations[-10:])

In [ ]:
print(df[['pl_rade', 'st_rad']].corr())

Expected strong correlation between stellar radius and planet radius, but found near-zero correlation. This suggests planet size is not strongly determined by star size alone across the full dataset — other factors dominate.


In [ ]:
print(df[['pl_rade', 'pl_bmasse']].corr())

In [ ]:
print(df[['pl_rade', 'pl_bmasse']].describe())

### ⚠️ Target Variable Anomaly Found — `pl_rade`

**Finding:** `pl_rade` max value was 5.14 TRILLION Earth radii — physically impossible.

The Astronomical Constants PDF states the maximum planet radius in this dataset is ~27.35 Jupiter radii = ~301 Earth radii.

Anything above 301 is corrupted observational data. **Next cell drops those rows.**

In [ ]:
df['pl_rade'].sort_values(ascending=False).head(20)

In [ ]:
print((df['pl_rade'] > 301).sum())

print(((df['pl_rade'] < 301) & (df['pl_rade'] > 0)).sum())

In [ ]:
df = df[df['pl_rade'] <= 301]
print(f'DataFrame shape after dropping rows: {df.shape}')

### ✅ `pl_rade` Cleaned

**Rows dropped:** 1,205 rows with impossible planet radius values (above 301 Earth radii).

**Rows remaining:** ~38,700+ rows — more than enough for reliable model training.

**`pl_bmasse` check:** Max value is 9,534 Earth masses — within physical range for massive gas giants. No rows dropped for mass.

In [ ]:
df['pl_bmasse'].sort_values(ascending=False).head(10)

In [ ]:
corr = df.corr(numeric_only=True)[['pl_rade', 'pl_bmasse']].sort_values('pl_rade', ascending=False)
print(corr)

In [ ]:
df['sy_mnum'].describe()

### 🗑️ `sy_mnum` Dropped — Zero Variance Column

**Finding:** `sy_mnum` (number of moons) has mean=0, std=0, min=0, max=0 — every single value is zero.

A column with zero variance gives the model zero information. **Dropped immediately.**

In [ ]:
corr = df.corr(numeric_only=True)[['pl_rade', 'pl_bmasse']].abs().sort_values('pl_rade', ascending=False)
print(corr)

In [ ]:
corr = df.corr(numeric_only=True)[['pl_rade', 'pl_bmasse']].abs().sort_values('pl_bmasse', ascending=False)
print(corr)

### 📋 Correlation Summary — Key Findings

**Strongest predictors of `pl_rade`:**
- `pl_eqt` (0.29) — equilibrium temperature
- `pl_insol` (0.155) — insolation flux
- `st_mass` (0.14) — stellar mass

**Strongest predictors of `pl_bmasse`:**
- `pl_orbeccen` (0.46) — orbital eccentricity
- `st_rad` (0.325) — stellar radius
- `sy_vmag` (-0.357) — visual magnitude

**Imputation decision:** No median imputation applied. Missing values left as NaN intentionally. Teammate should use **XGBoost or Random Forest** — both handle NaN natively without imputation bias.

In [ ]:
priority = ['pl_orbeccen', 'sy_w4mag', 'sy_w3mag', 'sy_w2mag', 'pl_eqt', 'pl_insol']
print(df[priority].isna().sum())

In [ ]:
print(df[['pl_eqt', 'st_teff', 'st_rad', 'pl_orbsmax']].isna().sum())

## 🔧 Step 7 — Feature Engineering

**What we are doing:** Creating 3 new columns from existing ones that give the model better information.

**Why these 3 features:**

1. `orbital_speed_proxy` = `pl_orbper / pl_orbsmax` — hints at how fast the planet orbits. Faster orbit = closer to star = more energy received.

2. `eccentricity_mass_interaction` = `pl_orbeccen × st_mass` — combines orbit shape with star mass. A stretched orbit around a heavy star creates stronger gravitational effects on the planet. **This is our innovative feature — achieved 0.48 correlation with `pl_bmasse`.**

3. `log_orbper` = log10(`pl_orbper`) — orbital periods range from 1 to 10,000+ days (huge skew). Log transformation compresses this for better model handling.

**Note to teammates:** Do NOT use `pl_ratror` — it is derived from `pl_rade` (target variable) and causes data leakage.

In [ ]:
df['orbital_speed_proxy'] = df['pl_orbper'] / df['pl_orbsmax']

In [ ]:
# Interaction between orbit shape and star mass
df['eccentricity_mass_interaction'] = df['pl_orbeccen'] * df['st_mass']

# Log of orbital period - compresses skewed distribution
df['log_orbper'] = np.log10(df['pl_orbper'])

In [ ]:
df[['orbital_speed_proxy','log_orbper','eccentricity_mass_interaction', 'pl_rade', 'pl_bmasse']].corr()[['pl_rade', 'pl_bmasse']]

### ✅ Feature Engineering Complete

**Final engineered features and their correlations:**

| Feature | pl_rade | pl_bmasse |
|---|---|---|
| `eccentricity_mass_interaction` | 0.043 | **0.483** |
| `log_orbper` | -0.088 | **0.318** |
| `orbital_speed_proxy` | 0.006 | **0.209** |

All three features show meaningful correlation with `pl_bmasse`. For `pl_rade`, original columns like `pl_eqt` remain the strongest predictors.

In [ ]:
df.shape

## 🏁 Handoff to Modeling Team

**Your job starts here.**

**What has been done (EDA + Cleaning):**
- Physics violations fixed: `sy_dist` (unit + sign), `pl_orbeccen` (1 invalid → NaN), `pl_orbincl` (1 invalid → NaN), `st_mass` (3 negatives → abs)
- 19 useless columns dropped (noisy, metadata, leakage)
- 1,205 rows with impossible `pl_rade` values dropped
- `sy_mnum` dropped (zero variance)
- Correlation analysis completed
- 3 engineered features added

**What you need to do:**
1. Use `pl_rade` and `pl_bmasse` as your TWO target variables (multi-output regression)
2. Use **XGBoost or Random Forest** — both handle NaN natively, no imputation needed
3. Train on all remaining columns EXCEPT `pl_rade` and `pl_bmasse`
4. Evaluate using MAE, RMSE, R² — compare at least 2 models
5. After predicting radius and mass, calculate escape velocity: `v_e = sqrt(2 * G * M / R)` where G=6.6743e-11, M=pl_bmasse×5.9722e24, R=pl_rade×6.371e6
6. DO NOT use `pl_ratror` — data leakage
7. DO NOT use any column ending in `err1` or `err2` as primary features — they are uncertainty measurements

**Final dataset shape after all cleaning:**